# Proyecto: 120 years of Olympic history
Analisis exploratorio de datos usando Pandas

**Integrantes:**
- Julian Andres Velandia Cortes

## 1. Cargar cada conjunto de datos 

In [ ]:
import pandas as pd

df_athlete = pd.read_csv('../data/athlete_events.csv')
df_regions = pd.read_csv('../data/noc_regions.csv')

## 2. Descripcion de las variables

**Dataset athlete_events.csv:**
* **ID:** Identificador de cada atleta.
* **Name:** Nombre del atleta.
* **Sex:** Sexo del atleta (M o F).
* **Age:** Edad del atleta en anios.
* **Height:** Estatura en centimetros.
* **Weight:** Peso en kilogramos.
* **Team:** Nombre del equipo o pais.
* **NOC:** Codigo de 3 letras del Comite Olimpico Nacional.
* **Games:** Anio y temporada de los juegos.
* **Year:** Anio de la edicion.
* **Season:** Temporada (Summer o Winter).
* **City:** Ciudad sede.
* **Sport:** Deporte general.
* **Event:** Prueba especifica en la que compitio.
* **Medal:** Medalla ganada (Gold, Silver, Bronze o vacio/NaN).

**Dataset noc_regions.csv:**
* **NOC:** Codigo de 3 letras del Comite Olimpico Nacional.
* **region:** Nombre del pais o region geografica.
* **notes:** Notas adicionales sobre la region o territorio.

## 3. Estadisticas basicas

In [ ]:
# 3. Genere las estadísticas básicas de cada dataframe y haga comparaciones
display(df_athlete.describe(include='all'))
display(df_regions.describe(include='all'))

**Comparacion entre los dataframes:**
El dataframe ""df_athlete"" tiene 271,116 filas y 15 columnas (combina variables numericas como edad, peso, estatura y ano con variables de texto), donde cada fila es la participacion de un atleta en una prueba y existen 226 codigos NOC distintos. El dataframe ""df_regions"" tiene 230 filas y 3 columnas (todas de texto), donde cada fila representa un codigo NOC unico con su respectiva region.

## 4. Creación de la variable IMC
Fórmula: Peso / (Estatura/100)^2

In [ ]:
# 4. Cree una nueva variable que se llame IMC
df_athlete['IMC'] = df_athlete['Weight'] / ((df_athlete['Height'] / 100) ** 2)
df_athlete[['Weight', 'Height', 'IMC']].head()

**¿Qué opina de esta nueva variable? ¿Tiene el mismo sentido en todos los deportes?**

El IMC sirve para darnos una idea general de la relacion entre el peso y la estatura de una persona, pero en atletas olimpicos no es una medida muy confiable para evaluar su estado fisico porque la formula no diferencia entre grasa y masa muscular.

Por eso mismo, no tiene el mismo sentido en todos los deportes. Cada disciplina exige un tipo de cuerpo muy distinto: por ejemplo, un atleta de levantamiento de pesas o de lucha va a tener un IMC muy alto (que en una tabla normal saldria como "sobrepeso"), pero en realidad ese peso es puro musculo necesario para su deporte. En cambio, deportistas de gimnasia o maraton suelen tener un IMC mucho mas bajo porque necesitan ser ligeros. Compararlos a todos con la misma regla del IMC no refleja su verdadero rendimiento ni su salud.

## 5. Fusion (merge) de atletas y regiones

In [ ]:
#fucionar el dataframe de atletas con el de regiones usando NOC
df_merged = pd.merge(df_athlete, df_regions, on='NOC', how='left')

# verificar el numero de filas antes y despues del cruce
print("Filas antes del merge:", len(df_athlete))
print("Filas despues del merge:", len(df_merged))

# revisar cuantos registros quedan sin region asociada
sin_region = df_merged['region'].isna().sum()
print("Filas sin region:", sin_region)
print("Codigos NOC sin region:", df_merged[df_merged['region'].isna()]['NOC'].unique()) #esto del .unique() es para obtener los valores distintos (sin repetir)

* **¿Por que cree que ocurre esto?** Quedan filas sin region porque en el dataset de atletas existen codigos NOC historicos o especiales (como "ROT" de refugiados o "TUV" de Tuvalu) que no tienen un pais asociado en la tabla pequena de regiones.
* **¿Que tipo de fusion (how) utilizo y por que?** Se utilizo "how='left'". Esto garantiza que se mantengan absolutamente todas las filas del dataframe principal de atletas sin perder ningun registro, trayendo la region cuando coincide el codigo NOC y dejando vacio ("NaN") donde no hay coincidencia.

## 6. Creacion de la variable binaria de medallas

In [ ]:
#preguntar si la columna 'Medal' NO esta vacia
#con el .astype(int) convertimos ese Verdadero/Falso en 1 y 0.
df_merged['tiene_medalla'] = df_merged['Medal'].notna().astype(int)

# mostrar  los primeros resultados para ver si quedo bien
df_merged[['Name', 'Sport', 'Medal', 'tiene_medalla']].head(10)

Con esta nueva columna binaria ("1" o "0"), podemos sumar facilmente cuantos podios ha conseguido un pais, un deporte o un atleta especifico utilizando operaciones basicas de Pandas, sin que nos estorben los valores vacios ("NaN").